In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [2]:
train_data = pd.read_csv('../data/model/train.csv')
train_labels = train_data['redemption_status'].values
train_data = train_data.drop(['id','customer_id','redemption_status'], axis=1)
valid_data = pd.read_csv('../data/model/valid.csv')
valid_labels = valid_data['redemption_status'].values
valid_data = valid_data.drop(['id','customer_id','redemption_status'], axis=1)

In [3]:
params = {}
params['label'] = train_labels
params['feature_name'] = list(train_data.columns)
train_matrix = lgb.Dataset(train_data.values, **params)
params = {}
params['label'] = valid_labels
params['feature_name'] = list(valid_data.columns)
valid_matrix = lgb.Dataset(valid_data.values, **params)

In [4]:
booster = {}
booster['boosting_type'] = 'gbdt'
booster['objective'] = 'binary'
booster['learning_rate'] = 0.01
booster['num_leaves'] = 24
booster['max_depth'] = 4
booster['max_bin'] = 256
booster['subsample'] = 0.5
booster['subsample_freq'] = 1
booster['colsample_bylevel'] = 0.5
booster['colsample_bytree'] = 0.5
booster['min_split_gain'] = 0.0
booster['min_sum_hessian'] = 1
booster['nthread'] = 3
booster['verbose'] = 0
booster['metric'] = 'auc'

In [5]:
params = {}
params['params'] = booster
params['train_set'] = train_matrix
params['valid_sets'] = [train_matrix, valid_matrix]
params['num_boost_round'] = 2000
params['early_stopping_rounds'] = 200
params['verbose_eval'] = 25

In [6]:
model = lgb.train(**params)

Training until validation scores don't improve for 200 rounds
[25]	training's auc: 0.961617	valid_1's auc: 0.954509
[50]	training's auc: 0.967738	valid_1's auc: 0.957078
[75]	training's auc: 0.970805	valid_1's auc: 0.958972
[100]	training's auc: 0.973153	valid_1's auc: 0.960197
[125]	training's auc: 0.975342	valid_1's auc: 0.962
[150]	training's auc: 0.976689	valid_1's auc: 0.962941
[175]	training's auc: 0.97813	valid_1's auc: 0.964489
[200]	training's auc: 0.979247	valid_1's auc: 0.965406
[225]	training's auc: 0.980403	valid_1's auc: 0.966122
[250]	training's auc: 0.981146	valid_1's auc: 0.967128
[275]	training's auc: 0.982022	valid_1's auc: 0.967615
[300]	training's auc: 0.982809	valid_1's auc: 0.967897
[325]	training's auc: 0.983593	valid_1's auc: 0.968063
[350]	training's auc: 0.984326	valid_1's auc: 0.968438
[375]	training's auc: 0.985018	valid_1's auc: 0.968524
[400]	training's auc: 0.98583	valid_1's auc: 0.968721
[425]	training's auc: 0.986525	valid_1's auc: 0.968879
[450]	train

In [7]:
model.save_model('../data/model/lightgbm_v1.model')

In [8]:
importance = model.feature_importance(importance_type='gain')
importance = pd.DataFrame(importance, columns=['importance'])
importance['feature'] = list(valid_data.columns)
importance['importance'] = importance['importance'] / importance['importance'].max()
importance = importance[['feature', 'importance']]
importance = importance.sort_values(by='importance', ascending=False)
importance = importance.reset_index(drop=True)

In [9]:
importance.head(10)

,feature,importance
0,cust_cdsc_cnt,1.000000
1,cust_cdsc_sum,0.598043
2,sum_trx_cust_coup_price,0.447950
3,max_trx_cust_coup_price,0.418356
4,cust_coup_prc,0.368429
5,cnt_coup_cdsc,0.313453
6,over_1,0.293968
7,cust_coup_cdsc,0.278345
8,cust_cdsc,0.273305
9,sum_coup_cdsc,0.251259
